In [1]:
import pandas as pd
from sklearn.linear_model import LassoCV, Lasso, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_validate

df = pd.read_csv('AmesHousing.csv')
df = df.drop(columns=df.select_dtypes(include=object))
df = df.dropna()

X = df.drop(columns='SalePrice')
Y = df['SalePrice']

pipe = Pipeline([
      ('norm',StandardScaler()),
      ('lasso',LassoCV(cv=10, random_state=0))
])

pipe.fit(X, Y)

lamb_opm = pipe['lasso'].alpha_
print(lamb_opm)

1091.6825619915269


In [2]:
model = Pipeline([
    ('norm',StandardScaler()),
    ('ridge',Lasso(alpha=lamb_opm))
])

cv = KFold(n_splits=10,shuffle=True,random_state=1)

results = cross_validate(model,X,Y,cv=cv,
                         scoring={'MSE':'neg_mean_squared_error',
                                  'RMSE':'neg_root_mean_squared_error',
                                  'MAE':'neg_mean_absolute_error',
                                  'R2':'r2'})

rmse = -results['test_RMSE']
r2 = results['test_R2']

print('Resultado via Lasso Regression:')
print(f'RMSE: {rmse.mean():.4f} ± {rmse.std():.4f}')
print(f'R²:   {r2.mean():.4f} ± {r2.std():.4f}')


### COMPARANDO COM LEAST SQUARES


model = Pipeline([
    ('OLS',LinearRegression())
])

cv = KFold(n_splits=10,shuffle=True,random_state=1)

results = cross_validate(model,X,Y,cv=cv,
                         scoring={'MSE':'neg_mean_squared_error',
                                  'RMSE':'neg_root_mean_squared_error',
                                  'MAE':'neg_mean_absolute_error',
                                  'R2':'r2'})

rmse = -results['test_RMSE']
r2 = results['test_R2']

print('\nResultado via Linear Regression:')
print(f'RMSE: {rmse.mean():.4f} ± {rmse.std():.4f}')
print(f'R²:   {r2.mean():.4f} ± {r2.std():.4f}')

Resultado via Lasso Regression:
RMSE: 35214.0927 ± 11947.8767
R²:   0.8085 ± 0.1326

Resultado via Linear Regression:
RMSE: 35413.4876 ± 12365.8708
R²:   0.8054 ± 0.1380
